In [1]:
import os, json, math, random, gc, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision.models import efficientnet_b0

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

print("✅ Imports successful")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

✅ Imports successful
✅ CUDA available: True


# BirdCLEF 2026 Training v6 - Perch + EfficientNet-B0 (Lean Ensemble)
## Strategy:
- **Perch** (specialized bird audio CNN)
- **EfficientNet-B0** (lightweight, efficient)
- Dropped ResNet18 for focus and speed
- 10 models total (5 folds × 2 architectures)
- Faster training, cleaner ensemble

In [ ]:
# === DATA PATHS & SPECIES ===
TRAIN_META_CSV = "/kaggle/input/competitions/birdclef-2026/train.csv"
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/train_audio"
TAXONOMY_CSV = "/kaggle/input/competitions/birdclef-2026/taxonomy.csv"

# Load complete species list from taxonomy.csv (234 species)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species = taxonomy_df["primary_label"].astype(str).tolist()

species_set = set(species)

print(f"✅ Loaded {len(species)} species from taxonomy.csv")
print(f"First 10 species: {species[:10]}")
print(f"Last 10 species: {species[-10:]}")

# Create species index mapping
idx = {lab: i for i, lab in enumerate(species)}
n_classes = len(species)

# Load training data
df = pd.read_csv(TRAIN_META_CSV)
print(f"\n✅ Loaded training data: {len(df)} samples")
print(f"Unique species in training: {df['primary_label'].nunique()}")

# Save species.json for inference
with open("/kaggle/working/species.json", "w") as f:
    json.dump(species, f)

print(f"✅ Saved species.json with {n_classes} species")

Number of species: 206
First 10 species: ['1161364', '116570', '1176823', '1595929', '209233', '22930', '22956', '22961', '22967', '22973']
✅ Species saved


In [3]:
# === CONFIG ===
CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
    epochs=15,
    lr=1e-3,
    patience=5,
    num_workers=4,
    persistent_workers=True,
    prefetch_factor=4,
    pin_memory=False,
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Config: {CFG}")

Config: {'sr': 16000, 'n_mels': 64, 'n_fft': 1024, 'hop': 320, 'fmin': 60, 'seconds': 5, 'batch_size': 32, 'epochs': 15, 'lr': 0.001, 'patience': 5, 'num_workers': 4, 'persistent_workers': True, 'prefetch_factor': 4, 'pin_memory': False, 'seed': 42, 'device': 'cuda'}


In [4]:
# === HELPER FUNCTIONS ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(len(species), dtype="float32")
    p = str(primary_id)
    if p in idx: y[idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[idx[sid]] = 1.0
    return y

def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target-len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("✅ Helper functions defined")

✅ Helper functions defined


In [5]:
# === PRECOMPUTE MELS FROM TRAIN_AUDIO ===
print("Precomputing mels from train_audio…")

MEL_OUT_DIR = "/kaggle/working/mels_v6"
os.makedirs(MEL_OUT_DIR, exist_ok=True)

for idx_, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
    except:
        print(f"Failed: {audio_path}")
        continue

    if sr0 != CFG["sr"]:
        y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])

    y = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
    mel = logmel_from_wave(y, CFG["sr"])

    np.save(
        Path(MEL_OUT_DIR) / (row["filename"].replace("/", "_") + ".npy"),
        mel
    )

print(f"✅ Mels saved from train_audio: {MEL_OUT_DIR}")

Precomputing mels from train_audio…


100%|██████████| 35549/35549 [43:18<00:00, 13.68it/s]

✅ Mels saved from train_audio: /kaggle/working/mels_v6


In [6]:
# === PSEUDO-LABELING ON TRAIN_SOUNDSCAPES ===
print("\n" + "="*70)
print("PSEUDO-LABELING: Extracting segments from train_soundscapes")
print("="*70)

SOUNDSCAPE_DIR = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
SOUNDSCAPE_ANNO_CSV = "/kaggle/input/competitions/birdclef-2026/train_soundscape_labels.csv"

try:
    soundscape_labels = pd.read_csv(SOUNDSCAPE_ANNO_CSV)
    print(f"Loaded {len(soundscape_labels)} soundscape annotations")
    
    # List to hold pseudo-labeled data
    pseudo_data = []
    
    for idx_, row in tqdm(soundscape_labels.iterrows(), total=len(soundscape_labels), desc="Processing soundscapes"):
        soundscape_id = row["soundscape_id"]
        audio_path = Path(SOUNDSCAPE_DIR) / f"{soundscape_id}.ogg"
        
        if not audio_path.exists():
            continue
        
        try:
            y, sr0 = sf.read(str(audio_path), always_2d=False)
        except:
            continue
        
        if sr0 != CFG["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])
        
        # Extract 5-second segments from soundscape
        segment_duration = CFG["seconds"]
        segment_samples = CFG["sr"] * segment_duration
        total_segments = max(1, len(y) // segment_samples)
        
        # Sample up to 3 segments per soundscape to avoid over-representation
        segment_indices = np.random.choice(total_segments, min(3, total_segments), replace=False)
        
        for seg_idx in segment_indices:
            start_sample = seg_idx * segment_samples
            end_sample = start_sample + segment_samples
            
            if end_sample > len(y):
                continue
            
            segment = y[start_sample:end_sample].astype(np.float32)
            segment = fixed_length_mono(segment, CFG["sr"], CFG["seconds"])
            mel = logmel_from_wave(segment, CFG["sr"])
            
            mel_name = f"{soundscape_id}_seg_{seg_idx}.npy"
            mel_path = Path(MEL_OUT_DIR) / mel_name
            np.save(mel_path, mel)
            
            # Create pseudo-label entry (use primary species from this soundscape)
            primary = row.get("primary_label", "unknown")
            secondary = row.get("secondary_labels", "[]")
            
            pseudo_data.append({
                "filename": mel_name,
                "primary_label": primary,
                "secondary_labels": secondary
            })
    
    if pseudo_data:
        pseudo_df = pd.DataFrame(pseudo_data)
        print(f"✅ Extracted {len(pseudo_df)} pseudo-labeled soundscape segments")
    else:
        print("⚠️  No soundscape segments extracted")
        pseudo_df = pd.DataFrame()
        
except Exception as e:
    print(f"⚠️  Soundscape processing failed: {e}")
    print("Continuing with train_audio only...")
    pseudo_df = pd.DataFrame()


PSEUDO-LABELING: Extracting segments from train_soundscapes
⚠️  Soundscape processing failed: [Errno 2] No such file or directory: '/kaggle/input/competitions/birdclef-2026/train_soundscape_labels.csv'
Continuing with train_audio only...


In [7]:
# === PERCH ARCHITECTURE (Lightweight CNN) ===
class PerchAudio(nn.Module):
    """Lightweight CNN optimized for bird audio"""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.head = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x

print("✅ Perch architecture defined")

✅ Perch architecture defined


In [8]:
# === EFFICIENTNET-B0 ARCHITECTURE ===
class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.model = efficientnet_b0(weights=None)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("✅ EfficientNet-B0 architecture defined")

✅ EfficientNet-B0 architecture defined


In [9]:
# === DATA AUGMENTATION DATASET ===
class ClipDatasetWithAugmentation(Dataset):
    def __init__(self, frame: pd.DataFrame, mel_root: str, cfg: dict, train: bool):
        self.df = frame.reset_index(drop=True)
        self.mel_root = Path(mel_root)
        self.cfg = cfg
        self.train = train
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1

    def apply_augmentation(self, mel):
        """Apply moderate augmentations"""
        if not self.train:
            return mel
        
        # SpecAugment: Time masking (30% chance)
        if np.random.rand() < 0.3:
            mask_width = np.random.randint(5, 15)
            mask_start = np.random.randint(0, max(1, mel.shape[1] - mask_width))
            mel[:, mask_start:mask_start+mask_width] *= np.random.uniform(0.3, 0.7)
        
        # SpecAugment: Frequency masking (30% chance)
        if np.random.rand() < 0.3:
            mask_height = np.random.randint(3, 8)
            mask_start = np.random.randint(0, max(1, mel.shape[0] - mask_height))
            mel[mask_start:mask_start+mask_height, :] *= np.random.uniform(0.3, 0.7)
        
        # Brightness (20% chance)
        if np.random.rand() < 0.2:
            brightness = np.random.uniform(0.9, 1.1)
            mel = mel * brightness
        
        # Noise (10% chance)
        if np.random.rand() < 0.1:
            noise = np.random.normal(0, 0.005, mel.shape)
            mel = mel + noise
        
        return mel

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        
        mel_name = r["filename"].replace("/", "_") + ".npy"
        mel_path = self.mel_root / mel_name
        mel = np.load(mel_path).astype("float32")
        
        T = mel.shape[1]
        W = self.win_frames
        
        if T <= W:
            pad = np.zeros((mel.shape[0], W - T), dtype=np.float32)
            mel = np.concatenate([mel, pad], axis=1)
        else:
            start = np.random.randint(0, T - W) if self.train else max(0, (T - W) // 2)
            mel = mel[:, start:start + W]
        
        mel = self.apply_augmentation(mel)
        mel = np.clip(mel, 0.0, 1.0).astype(np.float32)
        
        x = torch.from_numpy(mel)[None, ...]
        y = torch.from_numpy(row_to_multihot(r["primary_label"], r["secondary_labels"])).float()
        
        return x.float(), y

print("✅ Dataset class defined")

✅ Dataset class defined


In [10]:
# === PREPARE TRAINING DATA ===
device = torch.device(CFG["device"])

mel_root = Path(MEL_OUT_DIR)
available_mels = set(f.stem.replace('.npy', '') for f in mel_root.glob("*.npy"))

print(f"Available mel files: {len(available_mels)}")

train_df = df.copy()
train_df["filename"] = train_df["filename"].apply(lambda x: x.replace("/", "_"))
train_df = train_df[train_df["filename"].isin(available_mels)].reset_index(drop=True)

print(f"Training samples from train_audio: {len(train_df)}")

# Merge pseudo-labeled soundscape data if available
if len(pseudo_df) > 0:
    pseudo_df_copy = pseudo_df.copy()
    pseudo_df_copy["filename"] = pseudo_df_copy["filename"].str.replace('.npy', '')
    train_df = pd.concat([train_df, pseudo_df_copy], ignore_index=True)
    print(f"Added {len(pseudo_df)} pseudo-labeled soundscape segments")
    print(f"Total training samples: {len(train_df)}")
else:
    print("No soundscape pseudo-labels added")

species_counts = {sp: 0 for sp in species}
for idx_, row in train_df.iterrows():
    primary = str(row["primary_label"])
    if primary in species_counts:
        species_counts[primary] += 1
    secondary = row.get("secondary_labels", [])
    if pd.notna(secondary):
        for sp in parse_secondary(secondary):
            if sp in species_counts:
                species_counts[sp] += 1

n_classes = len(species)
print(f"\n✅ Training setup complete:")
print(f"  Device: {device}")
print(f"  Classes: {n_classes}")
print(f"  Samples: {len(train_df)}")
print(f"  Species with data: {sum(1 for c in species_counts.values() if c > 0)}")

Available mel files: 35549
Training samples from train_audio: 35549
No soundscape pseudo-labels added

✅ Training setup complete:
  Device: cuda
  Classes: 206
  Samples: 35549
  Species with data: 206


In [11]:
# === 5-FOLD CV TRAINING: PERCH ===
print("\n" + "="*70)
print("TRAINING: PerchAudio")
print("="*70)

all_scores_perch = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    print(f"\n🔄 Fold {fold_idx + 1}/5")
    
    train_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]
    
    train_ds = ClipDatasetWithAugmentation(train_fold, MEL_OUT_DIR, CFG, train=True)
    val_ds = ClipDatasetWithAugmentation(val_fold, MEL_OUT_DIR, CFG, train=False)
    
    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)
    
    model = PerchAudio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    
    pos_weight = torch.ones(n_classes).to(device)
    for i, sp in enumerate(species):
        pos_weight[i] = len(train_df) / (3.0 * max(1, species_counts[sp]))
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = AdamW(model.parameters(), lr=CFG["lr"])
    
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(CFG["epochs"]):
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
        
        if patience_counter >= CFG["patience"]:
            break
    
    model.load_state_dict(best_model_state)
    all_scores_perch.append(best_loss)
    torch.save(model.state_dict(), f"/kaggle/working/perch_v6_fold_{fold_idx}.pt")
    print(f"  Loss: {best_loss:.4f}")

print(f"\n✅ Perch Training Complete: {np.mean(all_scores_perch):.4f} ± {np.std(all_scores_perch):.4f}")


TRAINING: PerchAudio

🔄 Fold 1/5


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


  Loss: 0.7859

🔄 Fold 2/5
  Loss: 0.7661

🔄 Fold 3/5
  Loss: 0.8023

🔄 Fold 4/5
  Loss: 0.7384

🔄 Fold 5/5
  Loss: 0.7270

✅ Perch Training Complete: 0.7640 ± 0.0282


In [12]:
# === 5-FOLD CV TRAINING: EFFICIENTNET-B0 ===
print("\n" + "="*70)
print("TRAINING: EfficientNetB0Audio")
print("="*70)

all_scores_efficientnet = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    print(f"\n🔄 Fold {fold_idx + 1}/5")
    
    train_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]
    
    train_ds = ClipDatasetWithAugmentation(train_fold, MEL_OUT_DIR, CFG, train=True)
    val_ds = ClipDatasetWithAugmentation(val_fold, MEL_OUT_DIR, CFG, train=False)
    
    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)
    
    model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    
    pos_weight = torch.ones(n_classes).to(device)
    for i, sp in enumerate(species):
        pos_weight[i] = len(train_df) / (3.0 * max(1, species_counts[sp]))
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = AdamW(model.parameters(), lr=CFG["lr"])
    
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(CFG["epochs"]):
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f"/kaggle/working/efficientnet_v6_fold_{fold_idx}.pt")
        else:
            patience_counter += 1
            if patience_counter >= CFG["patience"]:
                break
    
    all_scores_efficientnet.append(best_loss)
    print(f"  Best Loss: {best_loss:.4f}")

print(f"\n✅ EfficientNet Training Complete: {np.mean(all_scores_efficientnet):.4f} ± {np.std(all_scores_efficientnet):.4f}")


TRAINING: EfficientNetB0Audio

🔄 Fold 1/5


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


  Best Loss: 0.7823

🔄 Fold 2/5
  Best Loss: 0.7781

🔄 Fold 3/5
  Best Loss: 0.8043

🔄 Fold 4/5
  Best Loss: 0.7423

🔄 Fold 5/5
  Best Loss: 0.7452

✅ EfficientNet Training Complete: 0.7704 ± 0.0236


In [13]:
# === COMPUTE OPTIMAL THRESHOLDS ===
print("\n" + "="*70)
print("THRESHOLD ANALYSIS - Per-Species Optimization (2-Model Ensemble)")
print("="*70)

from sklearn.metrics import f1_score

all_perch_preds = []
all_efficientnet_preds = []
all_targets = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    val_fold = train_df.iloc[val_idx]
    val_ds = ClipDatasetWithAugmentation(val_fold, MEL_OUT_DIR, CFG, train=False)
    val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)
    
    perch_model = PerchAudio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    perch_model.load_state_dict(torch.load(f"/kaggle/working/perch_v6_fold_{fold_idx}.pt"))
    perch_model.eval()
    
    eff_model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    eff_model.load_state_dict(torch.load(f"/kaggle/working/efficientnet_v6_fold_{fold_idx}.pt"))
    eff_model.eval()
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            
            perch_logits = perch_model(x)
            perch_probs = torch.sigmoid(perch_logits).cpu().numpy()
            
            eff_logits = eff_model(x)
            eff_probs = torch.sigmoid(eff_logits).cpu().numpy()
            
            all_perch_preds.append(perch_probs)
            all_efficientnet_preds.append(eff_probs)
            all_targets.append(y.numpy())

perch_preds = np.vstack(all_perch_preds)
eff_preds = np.vstack(all_efficientnet_preds)
targets = np.vstack(all_targets)

# Ensemble averaging
ensemble_preds = (perch_preds + eff_preds) / 2.0

# Per-species threshold optimization
optimal_thresholds = {}
for j, sp in enumerate(species):
    y_true = targets[:, j]
    y_score = ensemble_preds[:, j]
    
    if y_true.sum() == 0 or (1 - y_true).sum() == 0:
        optimal_thresholds[sp] = 0.5
        continue
    
    best_threshold = 0.5
    best_f1 = 0.0
    
    for threshold in np.arange(0.1, 0.9, 0.05):
        y_pred = (y_score >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    optimal_thresholds[sp] = best_threshold

with open("/kaggle/working/optimal_thresholds_v6.json", "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

print(f"✅ Computed {len(optimal_thresholds)} per-species thresholds")
print(f"Threshold range: [{min(optimal_thresholds.values()):.3f}, {max(optimal_thresholds.values()):.3f}]")
print(f"Mean threshold: {np.mean(list(optimal_thresholds.values())):.3f}")


THRESHOLD ANALYSIS - Per-Species Optimization (2-Model Ensemble)


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


✅ Computed 206 per-species thresholds
Threshold range: [0.100, 0.500]
Mean threshold: 0.290


In [14]:
# === TRAINING SUMMARY ===
print("\n" + "="*70)
print("TRAINING SUMMARY - v6 (Perch + EfficientNet-B0 Lean Ensemble)")
print("="*70)

print(f"\n📊 PERCH RESULTS:")
print(f"  Mean Val Loss: {np.mean(all_scores_perch):.4f} ± {np.std(all_scores_perch):.4f}")

print(f"\n📊 EFFICIENTNET-B0 RESULTS:")
print(f"  Mean Val Loss: {np.mean(all_scores_efficientnet):.4f} ± {np.std(all_scores_efficientnet):.4f}")

ensemble_aucs = []
for j in range(n_classes):
    y_true = targets[:, j]
    y_score = ensemble_preds[:, j]
    if y_true.sum() == 0 or (1 - y_true).sum() == 0:
        continue
    auc = roc_auc_score(y_true, y_score)
    ensemble_aucs.append(auc)

print(f"\n📊 2-MODEL ENSEMBLE (Perch + EfficientNet-B0):")
print(f"  Mean AUC: {np.mean(ensemble_aucs):.4f} ± {np.std(ensemble_aucs):.4f}")

print(f"\n✅ KEY FEATURES:")
print(f"  ✓ Perch (specialized bird audio)")
print(f"  ✓ EfficientNet-B0 (lightweight)")
print(f"  ✓ 10 total models (5 folds × 2 architectures)")
print(f"  ✓ Faster training than v5")
print(f"  ✓ Per-species threshold optimization")

print(f"\n💾 SAVED ARTIFACTS:")
print(f"  - Perch: perch_v6_fold_0-4.pt")
print(f"  - EfficientNet: efficientnet_v6_fold_0-4.pt")
print(f"  - Thresholds: optimal_thresholds_v6.json")
print(f"  - Species: species.json")

print(f"\n✅ Training Complete! Ready for 2-model ensemble inference.")


TRAINING SUMMARY - v6 (Perch + EfficientNet-B0 Lean Ensemble)

📊 PERCH RESULTS:
  Mean Val Loss: 0.7640 ± 0.0282

📊 EFFICIENTNET-B0 RESULTS:
  Mean Val Loss: 0.7704 ± 0.0236

📊 2-MODEL ENSEMBLE (Perch + EfficientNet-B0):
  Mean AUC: 0.5433 ± 0.0971

✅ KEY FEATURES:
  ✓ Perch (specialized bird audio)
  ✓ EfficientNet-B0 (lightweight)
  ✓ 10 total models (5 folds × 2 architectures)
  ✓ Faster training than v5
  ✓ Per-species threshold optimization

💾 SAVED ARTIFACTS:
  - Perch: perch_v6_fold_0-4.pt
  - EfficientNet: efficientnet_v6_fold_0-4.pt
  - Thresholds: optimal_thresholds_v6.json
  - Species: species.json

✅ Training Complete! Ready for 2-model ensemble inference.
